# Dimensionality Reduction — GOSH vs PIC cohort comparison
UMAP, t-SNE, and PCA of matched cohort feature spaces.
Each plot colours points by site (GOSH / PIC) to visualise distributional overlap.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import umap
import warnings
warnings.filterwarnings('ignore')

## 1. Load and prepare data

In [ ]:
# ── EDIT THESE PATHS ──────────────────────────────────────────────────────────
GOSH_WINDOWS  = 'data/processed/windows_matched.csv'
PIC_WINDOWS   = 'data/pic_data/windows_matched_pic.csv'
FEAT_COLS_CSV = 'data/models/transformer_bce/feature_cols.csv'  # or None

# Number of admissions to sample per site (None = use all)
# Reduce to ~500 per site if running on CPU to keep t-SNE fast
N_SAMPLE = 1000
SEED     = 42
# ─────────────────────────────────────────────────────────────────────────────

In [ ]:
# Load windows
print('Loading GOSH windows...')
gosh = pd.read_csv(GOSH_WINDOWS)
gosh.columns = gosh.columns.str.lower()

print('Loading PIC windows...')
pic  = pd.read_csv(PIC_WINDOWS)
pic.columns  = pic.columns.str.lower()

print(f'GOSH: {gosh["admission_id"].nunique():,} admissions, {len(gosh):,} windows')
print(f'PIC:  {pic["admission_id"].nunique():,} admissions,  {len(pic):,} windows')

In [ ]:
# Identify feature columns
NON_FEAT = {
    'admission_id', 'subject_id', 'step_index', 'label',
    'aki_stage', 'onset_dt', 'hours_to_onset', 'hours_since_admit',
    'window_start', 'window_end', 'split', 'exclude_early_onset',
    'baseline_creatinine', 'baseline_source', 'admit_dt',
    'discharge_dt', 'dob', 'has_aki', 'aki_detected', 'rank_from_end', 'proc_dialysis_rrt'
}

if FEAT_COLS_CSV:
    feat_cols = pd.read_csv(FEAT_COLS_CSV, header=None)[0].tolist()
    feat_cols = [c for c in feat_cols if c in gosh.columns and c in pic.columns]
else:
    shared = set(gosh.columns) & set(pic.columns)
    feat_cols = sorted([c for c in shared
                        if c not in NON_FEAT
                        and gosh[c].dtype in [np.float64, np.float32,
                                               np.int64,   np.int32]])

print(f'Feature columns: {len(feat_cols)}')

In [ ]:
# Build admission-level feature vectors
# Aggregate per admission: mean, max, last value
def admission_features(windows_df, feat_cols):
    """Aggregate window-level features to one vector per admission."""
    grp = windows_df.groupby('admission_id')[feat_cols]
    mean_ = grp.mean().add_suffix('_mean')
    max_  = grp.max().add_suffix('_max')
    last_ = grp.last().add_suffix('_last')
    adm   = pd.concat([mean_, max_, last_], axis=1).reset_index()
    # Carry through aki_stage and label
    meta  = windows_df.groupby('admission_id').agg(
        aki_stage = ('aki_stage', 'max'),
        label     = ('label',     'max'),
    ).reset_index()
    return adm.merge(meta, on='admission_id', how='left')

print('Building admission-level features...')
gosh_adm = admission_features(gosh, feat_cols)
pic_adm  = admission_features(pic,  feat_cols)

gosh_adm['site'] = 'GOSH'
pic_adm['site']  = 'PIC'

print(f'GOSH admissions: {len(gosh_adm):,}')
print(f'PIC  admissions: {len(pic_adm):,}')

In [ ]:
# Combine and align feature columns
combined = pd.concat([gosh_adm, pic_adm], ignore_index=True)

# Get all feature columns in combined (mean/max/last suffixed)
all_feat = [c for c in combined.columns
            if c not in {'admission_id', 'site', 'aki_stage', 'label'}]

# Impute missing with median (PIC may lack some GOSH-specific features)
combined[all_feat] = combined[all_feat].fillna(
    combined[all_feat].median()
)

# Optional: sample to speed up t-SNE
if N_SAMPLE:
    rng = np.random.default_rng(SEED)
    gosh_idx = combined[combined['site']=='GOSH'].sample(
        min(N_SAMPLE, (combined['site']=='GOSH').sum()),
        random_state=SEED).index
    pic_idx  = combined[combined['site']=='PIC'].sample(
        min(N_SAMPLE, (combined['site']=='PIC').sum()),
        random_state=SEED).index
    combined = combined.loc[list(gosh_idx) + list(pic_idx)].reset_index(drop=True)

X    = combined[all_feat].values.astype(np.float32)
site = combined['site'].values

# Standardise
scaler = StandardScaler()
X_sc   = scaler.fit_transform(X)

print(f'Combined matrix: {X_sc.shape}')

In [ ]:
# Colour and marker settings
SITE_COLOURS = {'GOSH': '#C0392B', 'PIC': '#2E75B6'}
SITE_MARKERS = {'GOSH': 'o',       'PIC': 's'}
ALPHA        = 0.55
S            = 18

def scatter_by_site(ax, coords, site, title):
    for s in ['GOSH', 'PIC']:
        mask = site == s
        ax.scatter(
            coords[mask, 0], coords[mask, 1],
            c      = SITE_COLOURS[s],
            marker = SITE_MARKERS[s],
            s      = S, alpha=ALPHA, label=s,
            linewidths=0,
        )
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_xticks([]); ax.set_yticks([])
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.legend(fontsize=10, markerscale=1.5, framealpha=0.8)

## 2. PCA

In [ ]:
print('Running PCA...')
pca     = PCA(n_components=2, random_state=SEED)
X_pca   = pca.fit_transform(X_sc)

var_exp = pca.explained_variance_ratio_ * 100

fig, ax = plt.subplots(figsize=(7, 6))
scatter_by_site(ax, X_pca, site,
                f'PCA  (PC1 {var_exp[0]:.1f}% | PC2 {var_exp[1]:.1f}%)')
ax.set_xlabel(f'PC1 ({var_exp[0]:.1f}%)', fontsize=11)
ax.set_ylabel(f'PC2 ({var_exp[1]:.1f}%)', fontsize=11)
ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout()
plt.savefig('exploratory_figures/pca_gosh_pic.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Total variance explained: {sum(var_exp):.1f}%')

## 3. UMAP

In [ ]:
print('Running UMAP (this may take 1-2 minutes)...')
reducer = umap.UMAP(
    n_components  = 2,
    n_neighbors   = 30,
    min_dist      = 0.1,
    metric        = 'euclidean',
    random_state  = SEED,
    verbose       = False,
)
X_umap = reducer.fit_transform(X_sc)

fig, ax = plt.subplots(figsize=(7, 6))
scatter_by_site(ax, X_umap, site, 'UMAP')
ax.set_xlabel('UMAP 1', fontsize=11)
ax.set_ylabel('UMAP 2', fontsize=11)
plt.tight_layout()
plt.savefig('exploratory_figures/umap_gosh_pic.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. t-SNE

In [ ]:
# Use PCA-50 as input to t-SNE (much faster, standard practice)
print('Running t-SNE (this may take 2-5 minutes)...')
tsne   = TSNE(
    n_components = 2,
    perplexity   = 30,
    max_iter       = 1000,
    random_state = SEED,
    verbose      = 1,
)
X_tsne = tsne.fit_transform(X_sc)

fig, ax = plt.subplots(figsize=(7, 6))
scatter_by_site(ax, X_tsne, site, 't-SNE  (perplexity=30)')
ax.set_xlabel('t-SNE 1', fontsize=11)
ax.set_ylabel('t-SNE 2', fontsize=11)
plt.tight_layout()
plt.savefig('exploratory_figures/tsne_gosh_pic.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Combined 3-panel figure

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(19, 6))

scatter_by_site(axes[0], X_pca,  site,
                f'(a) PCA\n{var_exp[0]:.1f}% + {var_exp[1]:.1f}% variance')
axes[0].set_xlabel('PC1', fontsize=10)
axes[0].set_ylabel('PC2', fontsize=10)

scatter_by_site(axes[1], X_umap, site, '(b) UMAP')
axes[1].set_xlabel('UMAP 1', fontsize=10)
axes[1].set_ylabel('UMAP 2', fontsize=10)

scatter_by_site(axes[2], X_tsne, site, '(c) t-SNE')
axes[2].set_xlabel('t-SNE 1', fontsize=10)
axes[2].set_ylabel('t-SNE 2', fontsize=10)

# Single shared legend
handles = axes[0].get_legend().legend_handles
for ax in axes:
    if ax.get_legend(): ax.get_legend().remove()
fig.legend(handles=handles, labels=['GOSH', 'PIC'],
           loc='lower center', ncol=2, fontsize=11,
           markerscale=1.5, framealpha=0.8,
           bbox_to_anchor=(0.5, -0.04))

n_gosh = (site == 'GOSH').sum()
n_pic  = (site == 'PIC').sum()
fig.suptitle(
    f'GOSH vs PIC feature space comparison  '
    f'(GOSH n={n_gosh:,}, PIC n={n_pic:,} admissions)',
    fontsize=12, y=1.01
)

plt.tight_layout(pad=1.5)
plt.savefig('exploratory_figures/dimensionality_reduction_combined.png',
            dpi=150, bbox_inches='tight')
plt.show()
print('Saved → dimensionality_reduction_combined.png')